# Experiment 001B v2.1: Gemma 2 Model-Scale Probe on Colab T4

This notebook is the Colab entrypoint for Stage 1B / Experiment 001B v2.1. It tests Gemma 2 text models under fixed prompt/generation length matrices, records OOM directly, and saves temporary CSV/figures under Google Drive.

Manual decode is used instead of `model.generate()` so `requested_gen_len` is controlled as an exact decode-step count.

## 1. Experiment Controls

Choose the model and benchmark matrix here. Colab Forms do not support a true multi-select dropdown, so prompt and generation lengths are exposed as native checkbox parameters. The selected matrix is `prompt_len choices x gen_len choices`.

In [ ]:
#@title 1.1 Experiment Controls
model_choice = "Gemma 2 2B IT" #@param ["Gemma 2 2B IT", "Gemma 2 9B IT", "Gemma 2 27B IT"]

# Available prompt_len options: 64, 128, 256, 512, 1024, 2048, 4096, 8192
prompt_lengths_text = "64,128,256,512,1024,2048" #@param {type:"string"}

# Available gen_len options: 16, 32, 64, 128
gen_lengths_text = "32,64,128" #@param {type:"string"}

repeat = 3 #@param {type:"integer"}
torch_dtype_name = "float16" #@param ["float16"]
save_results = True #@param {type:"boolean"}

PROMPT_LEN_OPTIONS = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
GEN_LEN_OPTIONS = [16, 32, 64, 128]


def parse_length_list(raw_text, valid_options, field_name):
    values = []
    for item in raw_text.split(','):
        item = item.strip()
        if not item:
            continue
        try:
            value = int(item)
        except ValueError as exc:
            raise ValueError(f'{field_name} contains a non-integer value: {item}') from exc
        if value not in valid_options:
            raise ValueError(f'{field_name} contains unsupported value: {value}. Valid options: {valid_options}')
        if value not in values:
            values.append(value)
    if not values:
        raise ValueError(f'Please provide at least one value for {field_name}.')
    return values

selected_prompt_lengths = parse_length_list(prompt_lengths_text, PROMPT_LEN_OPTIONS, 'prompt_lengths_text')
selected_gen_lengths = parse_length_list(gen_lengths_text, GEN_LEN_OPTIONS, 'gen_lengths_text')

print('Model choice:', model_choice)
print('Prompt lengths:', selected_prompt_lengths)
print('Generation lengths:', selected_gen_lengths)
print('Matrix size:', len(selected_prompt_lengths) * len(selected_gen_lengths))
print('Repeat:', repeat)
print('Dtype:', torch_dtype_name)
print('Save results:', save_results)

## 2. Drive Paths

Mount Google Drive and locate the project, model cache, temporary CSV directory, and temporary figure directory. Experiment 001B temporary outputs are written under `results/exp001B/temp/`.

In [ ]:
#@title 2.1 Mount Google Drive and Set Paths
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped or failed:', exc)

PROJECT_DIR = Path('/content/drive/MyDrive/EdgeLLM-Systems')
MODEL_DIR = PROJECT_DIR / 'models'
RESULT_DIR = PROJECT_DIR / 'results' / 'exp001B' / 'temp'
CSV_DIR = RESULT_DIR / 'csv'
FIGURE_DIR = RESULT_DIR / 'figures'
EXP_INFO_DIR = RESULT_DIR / 'exp_info'
RUN_CONFIG_DIR = EXP_INFO_DIR / 'run_config'
ENV_INFO_DIR = EXP_INFO_DIR / 'environment'
MODEL_INFO_DIR = EXP_INFO_DIR / 'model'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
ENV_INFO_DIR.mkdir(parents=True, exist_ok=True)
MODEL_INFO_DIR.mkdir(parents=True, exist_ok=True)

if not (PROJECT_DIR / 'edge_llm_systems').exists():
    raise FileNotFoundError(
        f'Cannot find project package at {PROJECT_DIR / "edge_llm_systems"}. '
        'Please sync the repository to MyDrive/EdgeLLM-Systems first.'
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR:    ', PROJECT_DIR)
print('MODEL_DIR:      ', MODEL_DIR)
print('CSV_DIR:        ', CSV_DIR)
print('FIGURE_DIR:     ', FIGURE_DIR)
print('EXP_INFO_DIR:   ', EXP_INFO_DIR)
print('RUN_CONFIG_DIR: ', RUN_CONFIG_DIR)
print('ENV_INFO_DIR:   ', ENV_INFO_DIR)
print('MODEL_INFO_DIR: ', MODEL_INFO_DIR)

## 3. Dependencies

Install the runtime packages needed by the benchmark and plotting code. Run this once after a fresh Colab runtime starts.

In [ ]:
#@title 3.1 Install / Check Dependencies
!pip install -q transformers accelerate sentencepiece huggingface_hub pyyaml pandas matplotlib

## 4. Hugging Face Access

Gemma weights may require Hugging Face access approval. This cell reads the token from Colab Secrets instead of prompting for manual input. Recommended secret name: `HF_TOKEN`.

In [ ]:
#@title 4.1 Hugging Face Access from Colab Secrets
use_hf_token_from_secrets = True #@param {type:"boolean"}
hf_secret_name = "HF_TOKEN" #@param {type:"string"}

if use_hf_token_from_secrets:
    try:
        from google.colab import userdata
        from huggingface_hub import login

        hf_token = userdata.get(hf_secret_name)
        if hf_token:
            login(token=hf_token)
            print(f'Hugging Face login completed from Colab Secret: {hf_secret_name}')
        else:
            print(f'No token found in Colab Secret: {hf_secret_name}. Cached models may still load.')
    except Exception as exc:
        print('Hugging Face secret login skipped or failed:', exc)
else:
    print('HF login skipped. Cached models can still be loaded from MODEL_DIR.')

## 5. Imports And Environment

Import reusable project modules and print the execution environment. Keep this output for the experiment log.

In [ ]:
#@title 5.1 Import Modules and Record Environment
import json
import platform
import pandas as pd
import torch
import transformers
import yaml

from benchmarks.profiling.run_exp001_v2_1 import (
    build_test_configs,
    choose_warmup_gen_len,
    dtype_from_config,
    make_load_status_row,
    measure_single_v2_1,
    run_matrix,
    write_csv,
)
from edge_llm_systems.cuda_utils import cleanup_cuda, require_cuda
from edge_llm_systems.model_registry import get_model_spec
from edge_llm_systems.model_utils import inspect_causal_lm
from edge_llm_systems.models import load_causal_lm
from scripts.plot_exp001B_v2_1 import plot_exp001B_v2_1

cuda_available = torch.cuda.is_available()
environment_info = {
    'project_dir': str(PROJECT_DIR),
    'model_dir': str(MODEL_DIR),
    'csv_dir': str(CSV_DIR),
    'figure_dir': str(FIGURE_DIR),
    'exp_info_dir': str(EXP_INFO_DIR),
    'run_config_dir': str(RUN_CONFIG_DIR),
    'environment_dir': str(ENV_INFO_DIR),
    'model_info_dir': str(MODEL_INFO_DIR),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'pytorch': torch.__version__,
    'transformers': transformers.__version__,
    'pandas': pd.__version__,
    'cuda_available': cuda_available,
    'cuda_runtime': torch.version.cuda,
}

if cuda_available:
    props = torch.cuda.get_device_properties(0)
    environment_info.update({
        'gpu': torch.cuda.get_device_name(0),
        'cuda_capability': torch.cuda.get_device_capability(0),
        'gpu_memory_gb': round(props.total_memory / 1024**3, 2),
    })

for key, value in environment_info.items():
    print(f'{key}: {value}')

require_cuda()

## 6. Runtime Config

Build the runtime configuration from the selected model and selected prompt/gen lengths. Output filenames include the model, selected lengths, and timestamp.

In [ ]:
#@title 6.1 Build v2.1 Runtime Config
from datetime import datetime

prompt_lengths = list(selected_prompt_lengths)
gen_lengths = list(selected_gen_lengths)

if not prompt_lengths:
    raise ValueError('Please select at least one prompt_len value in Cell 1.1.')
if not gen_lengths:
    raise ValueError('Please select at least one gen_len value in Cell 1.1.')

selected_model = get_model_spec(model_choice)
model_slug = selected_model.slug
run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')
prompt_tag = '-'.join(str(x) for x in prompt_lengths)
gen_tag = '-'.join(str(x) for x in gen_lengths)

output_stem = f'exp001B_v2_1_{model_slug}_p{prompt_tag}_g{gen_tag}_{run_tag}'
output_csv = CSV_DIR / f'{output_stem}.csv'
output_figure = FIGURE_DIR / f'{output_stem}.png'
run_config_path = RUN_CONFIG_DIR / f'{output_stem}_run_config.json'
environment_info_path = ENV_INFO_DIR / f'{output_stem}_environment.json'
model_info_path = MODEL_INFO_DIR / f'{model_slug}_model_info.json'
config = {
    'experiment_name': 'exp001B_v2.1_gemma2_model_scale_t4',
    'model_choice': model_choice,
    'model_id': selected_model.model_id,
    'model_slug': model_slug,
    'model_dir': str(MODEL_DIR),
    'prompt_lengths': prompt_lengths,
    'gen_lengths': gen_lengths,
    'repeat': int(repeat),
    'torch_dtype': torch_dtype_name,
    'device_map': {'': 0},
    'base_prompt': 'The quick brown fox jumps over the lazy dog. ',
    'output_csv': str(output_csv),
    'output_figure': str(output_figure),
    'run_config_path': str(run_config_path),
    'environment_info_path': str(environment_info_path),
    'model_info_path': str(model_info_path),
}
run_config_record = {
    **config,
    'matrix_configs': build_test_configs(prompt_lengths, gen_lengths),
}
environment_record = {
    **environment_info,
    'experiment_name': config['experiment_name'],
    'model_choice': config['model_choice'],
    'model_id': config['model_id'],
    'model_slug': config['model_slug'],
    'run_tag': run_tag,
    'output_csv': str(output_csv),
    'output_figure': str(output_figure),
}

with run_config_path.open('w', encoding='utf-8') as f:
    json.dump(run_config_record, f, ensure_ascii=False, indent=2)
with environment_info_path.open('w', encoding='utf-8') as f:
    json.dump(environment_record, f, ensure_ascii=False, indent=2)

print('Selected model:', selected_model.choice)
print('Hugging Face model_id:', selected_model.model_id)
print('Family:', selected_model.family)
print('Size label:', selected_model.size_label)
print('Notes:', selected_model.notes)
print('Prompt lengths:', prompt_lengths)
print('Generation lengths:', gen_lengths)
print('Matrix configs:', build_test_configs(prompt_lengths, gen_lengths))
print('Device map:', config['device_map'])
print('Output CSV:', output_csv)
print('Output figure:', output_figure)
print('Run config JSON:', run_config_path)
print('Environment JSON:', environment_info_path)
print('Model info JSON:', model_info_path)

## 7. Model Loading

Load tokenizer and model from the persistent Google Drive model cache when available. If loading OOMs, the notebook writes an OOM CSV row instead of changing the experiment matrix.

In [ ]:
#@title 7.1 Load Model, Inspect Metadata, and Save Model Info
MODEL_LOADED = False
model_info = {}
model = None
tokenizer = None
device = None
model_path = None
LOAD_OOM_MESSAGE_ZH = (
    '\u6a21\u578b\u52a0\u8f7d\u9636\u6bb5\u53d1\u751f CUDA '
    '\u663e\u5b58\u4e0d\u8db3\uff1b\u672c\u6b21\u4e0d\u81ea\u52a8'
    '\u964d\u7ea7\u6a21\u578b\u6216\u7f29\u5c0f prompt_len\uff0c'
    '\u76f4\u63a5\u8bb0\u5f55\u4e3a OOM\u3002'
)
LOAD_ERROR_PREFIX_ZH = '\u6a21\u578b\u52a0\u8f7d\u5931\u8d25\uff1a'

try:
    tokenizer, model, model_path = load_causal_lm(
        model_id=config['model_id'],
        model_dir=config['model_dir'],
        torch_dtype=dtype_from_config(config['torch_dtype']),
        device_map=config['device_map'],
    )
    device = next(model.parameters()).device
    model_info = inspect_causal_lm(model)
    model_info_record = {
        'experiment_name': config['experiment_name'],
        'model_choice': config['model_choice'],
        'model_id': config['model_id'],
        'model_slug': config['model_slug'],
        'model_path': str(model_path),
        **model_info,
    }
    MODEL_LOADED = True

    print('Model loaded:', config['model_id'])
    print('Model path:', model_path)
    print('Device:', device)
    print('Model info:')
    for key, value in model_info_record.items():
        print(f'  {key}: {value}')

    model_info_path = Path(config['model_info_path'])
    if model_info_path.exists():
        print('Model info JSON already exists; skip saving:', model_info_path)
    else:
        model_info_path.parent.mkdir(parents=True, exist_ok=True)
        with model_info_path.open('w', encoding='utf-8') as f:
            json.dump(model_info_record, f, ensure_ascii=False, indent=2)
        print('Model info JSON saved:', model_info_path)

except torch.cuda.OutOfMemoryError:
    cleanup_cuda()
    message = LOAD_OOM_MESSAGE_ZH
    row = make_load_status_row(status='oom', message_zh=message)
    if save_results:
        write_csv(output_csv, [row])
    print(message)
    print('CSV saved:', output_csv)

except RuntimeError as exc:
    cleanup_cuda()
    if 'out of memory' in str(exc).lower():
        message = LOAD_OOM_MESSAGE_ZH
        status = 'oom'
    else:
        message = f'{LOAD_ERROR_PREFIX_ZH}{exc}'
        status = 'error'
    row = make_load_status_row(status=status, message_zh=message)
    if save_results:
        write_csv(output_csv, [row])
    print(message)
    print('CSV saved:', output_csv)

## 8. Warm-Up

Run one small warm-up configuration to reduce first-run CUDA/kernel initialization noise. Warm-up is not saved to CSV.

In [ ]:
#@title 8.1 Warm Up GPU
if MODEL_LOADED:
    warmup_gen_len = choose_warmup_gen_len(config['gen_lengths'])
    print(f'Warm-up gen_len: {warmup_gen_len}')
    for warmup_prompt_len in config['prompt_lengths']:
        try:
            _ = measure_single_v2_1(
                warmup_prompt_len,
                warmup_gen_len,
                model,
                tokenizer,
                device,
                config['base_prompt'],
            )
            print(f'Warm-up completed: prompt={warmup_prompt_len}, gen={warmup_gen_len}')
        except torch.cuda.OutOfMemoryError:
            cleanup_cuda()
            print(f'Warm-up OOM: prompt={warmup_prompt_len}, gen={warmup_gen_len}; formal test will continue.')
        except RuntimeError as exc:
            cleanup_cuda()
            print(f'Warm-up failed: prompt={warmup_prompt_len}, gen={warmup_gen_len}; error={exc}')
else:
    print('Warm-up skipped because the model was not loaded.')

## 9. Benchmark Matrix

Run every selected `prompt_len x gen_len` pair. OOM does not shrink the matrix; failed configurations are kept as CSV rows with `status=oom`.

In [ ]:
#@title 9.1 Run v2.1 Benchmark Matrix
def format_row_summary(row):
    return (
        f"prompt={row['prompt_len']}, gen={row['requested_gen_len']} | "
        f"TTFT={row['ttft_ms']} ms | TPOT={row['tpot_ms']} ms | "
        f"tok/s={row['tokens_s']} | peak={row['peak_mem_mb']} MB | "
        f"KV={row['kv_pkv_final_mb']} MB | KV/peak={row['kv_peak_pct']}% | "
        f"repeat={row['repeat_completed']} | status={row['status']}"
    )


def print_progress(event):
    event_name = event.get('event')
    if event_name == 'matrix_start':
        total = event['config_total']
        repeat_total = event['repeat']
        print(f'Start matrix: {total} configs, repeat={repeat_total}, total measured runs={total * repeat_total}')
    elif event_name == 'config_start':
        print(
            f"[{event['config_index']}/{event['config_total']}] "
            f"Start config: prompt_len={event['prompt_len']}, gen_len={event['requested_gen_len']}"
        )
    elif event_name == 'repeat_start':
        print(f"  - repeat {event['repeat_index']}/{event['repeat_total']} ...")
    elif event_name == 'repeat_done':
        row = event['row']
        print(
            f"  - repeat {event['repeat_index']}/{event['repeat_total']} done: "
            f"TTFT={row['ttft_ms']} ms, TPOT={row['tpot_ms']} ms, "
            f"peak={row['peak_mem_mb']} MB, KV%={row['kv_peak_pct']}"
        )
    elif event_name == 'config_done':
        row = event['row']
        print(f"[{event['config_index']}/{event['config_total']}] Averaged result:")
        print('  ' + format_row_summary(row))
    elif event_name == 'config_oom':
        row = event['row']
        print(f"[{event['config_index']}/{event['config_total']}] OOM recorded:")
        print('  ' + format_row_summary(row))
        print('  message:', row['message_zh'])
    elif event_name == 'config_error':
        row = event['row']
        print(f"[{event['config_index']}/{event['config_total']}] ERROR recorded:")
        print('  ' + format_row_summary(row))
        print('  message:', row['message_zh'])
    elif event_name == 'matrix_done':
        print(f'Matrix done: {event["config_total"]} configs completed or recorded.')

if MODEL_LOADED:
    rows = run_matrix(
        prompt_lengths=config['prompt_lengths'],
        gen_lengths=config['gen_lengths'],
        model=model,
        tokenizer=tokenizer,
        device=device,
        repeat=config['repeat'],
        base_prompt=config['base_prompt'],
        progress_callback=print_progress,
    )

    if save_results:
        write_csv(output_csv, rows)
        print('CSV saved:', output_csv)

    df = pd.DataFrame(rows)
    display(df)
else:
    print('Model was not loaded. Check Cell 7.1 output and the saved load-status CSV if available.')

## 10. Plot Results

Generate a v1-style overview figure from the saved CSV. Only `status=ok` rows are plotted; OOM rows remain in the CSV.

In [ ]:
#@title 10.1 Plot v2.1 Results
if save_results and 'output_csv' in globals() and output_csv.exists():
    plot_exp001B_v2_1(
        csv_path=output_csv,
        output_path=output_figure,
        model_label=config['model_choice'],
    )
    print('Figure saved:', output_figure)
else:
    print('No saved CSV found. Run Cell 9.0 with save_results=True before plotting.')

## 11. Cleanup

Release local Python references and clear unused CUDA cache blocks before switching models or ending the session.

In [ ]:
#@title 11.1 Release CUDA Cache
try:
    del model
except Exception:
    pass
cleanup_cuda()
print('CUDA cache cleanup completed.')